In [7]:
#####
# This code dose the following: 
# 1. connects to dbGaP pilot server
# 2. gets all subjects' phenotype data set for the ICAC study.
# 3. Reports the data in a csv file. The data will be in similar format as here: https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/dataset.cgi?study_id=phs002921.v2.p1&pht=12614
#
# 
import os
import requests
import csv
from datetime import datetime
import pytz
#from tzlocal import get_localzone
from time import sleep
from fhir_fetcher import fetch_all_data  # Ensure this module is available and handles paging through all records

def fetch_patient_observations(session, fhir_base_url, patient_id):
    qstr = f'Observation?subject=Patient/{patient_id}'
    start_url = f"{fhir_base_url}/{qstr}"
    observations = fetch_all_data(session, start_url, 0)  # Fetch all observations for the patient
    return observations

def extract_observation_data(observations):
    data = {}
    for entry in observations:
        resource = entry.get('resource', {})
        code = resource.get('code', {}).get('coding', [{}])[0]
        attribute_name = code.get('display', '')
        value_string = resource.get('valueString', '')
        value_quantity = resource.get('valueQuantity', {}).get('value', '')
        if attribute_name:
            value = value_string if value_string else value_quantity
            if value:
                data[attribute_name] = value
    return data

def fetch_patient_ids(session, fhir_base_url, study_reference):
    query_url = f"{fhir_base_url}/ResearchSubject?study={study_reference}"
    print (query_url)
    research_subjects = fetch_all_data(session, query_url, 0)
    patient_ids = [entry['resource']['individual']['reference'].split('/')[-1] for entry in research_subjects]
    return patient_ids

def main():
     # Get the local server time
     
    starttime = datetime.now()
    starttimeStr = starttime.strftime('%Y-%m-%d %H:%M:%S')
    print("====== In server local time zone: start time:", starttimeStr)

    # Convert local time to EST/EDT
    eastern = pytz.timezone('US/Eastern')
    eastern_time = starttime.astimezone(eastern)
    eastern_time_str = eastern_time.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    print(f"====== EST/EDT start time: {eastern_time_str}")
    
    ###############################################################################################
    # get the token from https://www.ncbi.nlm.nih.gov/gap/power-user-portal/.  
    #  Scroll down and click on the "Task Specific Token" button to get the light-weight version of the dbGaP RAS Passport.
    #  Save the file into a text. In my example, it is saved to .task-specific-token_all.txt. 
    ###############################################################################################
    TST_PATH = 'task-specific-token-all.txt'  
    # fhir_base_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot1/x1"
    fhir_base_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1"
                   
    with open(os.path.expanduser(TST_PATH), 'r') as f:  
        tst_token = f.read().strip()
    
    session = requests.Session()
    session.headers.update({
        'Accept': 'application/fhir+json',
        'Authorization': f'Bearer {tst_token}',
        'Content-Type': 'application/x-www-form-urlencoded',
    })

    study_reference = "phs002921"

    ################################################################################################
    # https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/ResearchSubject?study=phs002921
    # https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Observation?subject=Patient/4317770
    # ###############################################################################################
    
    patient_ids = fetch_patient_ids(session, fhir_base_url, study_reference)
    print(f"Total patients fetched: {len(patient_ids)}")

    data = []
    columns = set()
    patients_with_observations = 0
    for patient_id in patient_ids[:10]:
        observations = fetch_patient_observations(session, fhir_base_url, patient_id)
        observation_data = extract_observation_data(observations)
        if observation_data:
            observation_data['Patient'] = patient_id
            columns.update(observation_data.keys())
            data.append(observation_data)
            patients_with_observations += 1
            # print(f"Observations obtained for patient: {patient_id}")
            print(f"Number of patients with observations obtained: {patients_with_observations}")

        sleep(0.5)  # Add a delay of 1 second between each patient API request to avoid rate limits

    columns = ['Patient'] + sorted(columns)  # Ensure 'Patient' is the first column

    output_file = 'patient_observations.csv'
    with open(output_file, 'w', newline='') as csvfile:
        csvwriter = csv.DictWriter(csvfile, fieldnames=columns)
        csvwriter.writeheader()
        csvwriter.writerows(data)

    print(f"Data written to {output_file}")

    endtime = datetime.now()
    endtimeStr = endtime.strftime('%Y-%m-%d %H:%M:%S')
    print("====== end time:", endtimeStr)

    elapsed_time = endtime - starttime
    elapsed_seconds = elapsed_time.total_seconds()
    eminutes = elapsed_seconds // 60
    eseconds = elapsed_seconds % 60

    print(f"===========Elapsed time: {int(eminutes)} minutes and {int(eseconds)} seconds.")

if __name__ == "__main__":
    main()


====== In server local time zone: start time: 2025-01-07 16:12:02
====== EST/EDT start time: 2025-01-07 16:12:02 EST-0500
https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/ResearchSubject?study=phs002921
Total patients fetched: 1035
Number of patients with observations obtained: 1
Number of patients with observations obtained: 2
Number of patients with observations obtained: 3
Number of patients with observations obtained: 4
Number of patients with observations obtained: 5
Number of patients with observations obtained: 6
Number of patients with observations obtained: 7
Number of patients with observations obtained: 8
Number of patients with observations obtained: 9
Number of patients with observations obtained: 10
Data written to patient_observations.csv
====== end time: 2025-01-07 16:13:45
===========Elapsed time: 1 minutes and 42 seconds.
